In [ ]:
@dataclass
class Const:
    sym : str

@dataclass
class PatVar:
    v : str

@dataclass
class MetaVar:
    v : str

@dataclass
class App:
    f : Term
    x : Term

type Term = App | Const | PatVar | MetaVar




What makes this not a general term rewriting system keyed on outer f?

lfdtt vs lfhol


In [ ]:
from dataclasses import dataclass


@dataclass
class App:
    f : str
    args : list["Term"]

@dataclass
class Var:
    v : str

type Term = App | Var
type Subst = dict[str, App]
def pmatch(pat : Term, t : App) -> Subst | None:
    subst = {}
    todo = [(pat,t)]
    while todo:
        pat,t = todo.pop()
        match pat:
            case Var(v):
                t1 = subst.get(v)
                if t1 is not None:
                    if t != t1:
                        return None
                else:
                    subst[v] = t
            case App(f,args):
                if t.f != f or len(args) != len(t.args):
                    return None
                else:
                    todo.extend(zip(args, t.args))
    return subst

def substitute(p : Term, subst : Subst) -> App:
    match p:
        case Var(v):
            return subst[v]
        case App(f, args):
            return App(f, list(map(lambda p: substitute(p, subst), args)))

@dataclass
class DefCase:
    pat : Term # n-arity?
    rhs : Term

type Prog = dict[str, list[DefCase]]

def apply_defn(t : Term, defns : list[DefCase]) -> Term | None:
    for defn in defns:
        subst = pmatch(defn.pat, t)
        if subst is not None:
            return substitute(defn.rhs, subst)
    else:
        return None






def interp(p : Prog, t : Term) -> Term:
    args = map(interp(p), t.args)

def succ(x): return App("succ", [x])
zero = App("zero", [])
x = Var("x")

prog = {
    "double" : [
        DefCase(succ(x), succ(succ(x))),
        DefCase(zero, zero)
    ]
}

interp(prog, zero)


In [ ]:

class Defn:
    name : str
    var : str # foo (x : VTyp) : OutType
    vtyp : Term
    typ : Type
    cases : list[DepCase]

class DepCase
    pat : Term
    rhs : Term
    typ : Term
type depprog = dict[str, ]

Why not do agda style definitions in knuckeldragger.
Do termination check
Require 

```
app : {n} -> Vec n a -> Vec n a

```


Pattern matching - what is it. Patterns are sets. But also might patterns be setoids, or groupoids or subsets akin to how types are sets? If we're doing AC matching or some other equational matching, we're really trying to match into a quotient construction.



Try to write out raw version
vampire query_answering?



Agda is kind of close to lambda free dtt?
What about kind of a shallow agda frontend that recompiles to the well formed proof obligations


atomic type theory
Named nones
What about a model where everything has None in it?
The "var" abstraction. Could be seen as an abstract interpretation. Constant propagation. None = dunno

Dependent egraph ignoring hash cons becomes dependent 

Lambda free dependent types is roughly GATs? Which is roughly essentialy algebraic theories?
Which is roughly 

Lazi

https://people.cs.nott.ac.uk/psztxa/publ/fomus19.pdf naive type theory



What is the lightest weight way to have implicits?
Keyword arguments and bespoke roeutines for every combo that looks through children.
But sometimes we want to know about our parents

Toss up into a generator effect system?
Generate side constraints?  t, contrs
Everything needs to accept this tuple

make a concrete new dsl


Basically:
1. wrap in class
2. subclass
3. monkey patch tag
4. registry
5. lark dsl

Those are the things python affords us

Elaboration
Ulf Norell

https://www.andrew.cmu.edu/user/avigad/Papers/constr.pdf Elaboration in Dependent Type Theory

This is similar to the FreshVar mechanism, but it isn't kernel _persay_. We are embedding a diffetent notion of thing into consts using some kind of goofy ad hoc thing.

Named metavars are also possible just by making a constant that starts with ?
That's kind of fun

Add exists to proofstate



In [18]:
ipy = get_ipython()
#ipy.set_next_input('', replace=False)
ipy.execution_count
In[ipy.execution_count - 1]
#Out

"ipy = get_ipython()\n#ipy.set_next_input('', replace=False)\nipy.execution_count\nIn[ipy.execution_count - 1]\n#Out"

In [ ]:
Hello world

In [23]:
ast.dump(ast.parse("1 + 2", mode="eval"))

'Expression(body=BinOp(left=Constant(value=1), op=Add(), right=Constant(value=2)))'

In [22]:
mod = ast.parse("""
match x:
    case Foo(Biz(x),x):
        print("one")          
    case _:
        print("two")
    case None:
        print("three")
          """, mode="eval")
print(ast.dump(mod, indent=4))

SyntaxError: invalid syntax (<unknown>, line 2)

I guess I could add preconditions to the path_condition... But they are already there. Pattern matching doesn't "expose" anything. I suppose I could have kd.Proof inside body that somehow propagate into by field of contract call...


() = my_theorem

Or overload matching on my_theorem... to pretend it's kind of `() : my_theorem`


In [ ]:
import ast

mod = ast.parse("""
match x:
    case Foo(Biz(x),x):
        print("one")          
    case 1:
        print("two")
          """)


def reflect_pattern(subject : object, pattern : ast.pattern, locals) -> tuple[list[smt.BoolRef], dict[str, object]]:
    match pattern:
        case ast.MatchClass(cls=ast.Name(id=constr_name), patterns=patterns, kwd_attrs=kwd_attrs, kwd_patterns=kwd_patterns):
            assert not kwd_attrs and not kwd_patterns, "Keyword patterns not supported yet"
            assert isinstance(subject, smt.ExprRef), "Expected an SMT expression as scrutinee"
            sort = subject.sort()
            assert isinstance(sort, smt.DatatypeSortRef), "Expected a datatype sort"
            for nconstr in range(sort.num_constructors()):
                constr = sort.constructor(nconstr)
                path_cond = [sort.recognizer(nconstr)(subject)]
                if constr_name == constr.name():
                    assert len(patterns) == constr.arity(), "Constructor arity mismatch"
                    for nfield, pattern in enumerate(patterns):
                        cond, locals = reflect_pattern(pattern, sort.accessor(nconstr,nfield))(subject), locals)
                        path_cond.extend(cond)
                    return path_cond, locals
            else:
                raise ValueError(f"Constructor {constr_name} not found in sort {s}")
        case ast.MatchAs(pattern=p, identifier=v):
            # Todo check for repeats names?
            if pattern is not None:
                cond, locals = reflect_pattern(subject, p, locals)
            else:
                cond = []
            if v is not None:
                locals = {**locals, v: subject}
            return cond, locals
        case ast.MatchValue(value):
            return [subject == value], locals
        case _:
            # MatchOr
            # MatchMappin
            # MatchStar
            # MatchSingleton
            # matchSequence
            raise NotImplementedError(f"Unsupported pattern: {ast.dump(pattern)}")

"""
    pattern = MatchValue(expr value)
            | MatchSingleton(constant value)
            | MatchSequence(pattern* patterns)
            | MatchMapping(expr* keys, pattern* patterns, identifier? rest)
            | MatchClass(expr cls, pattern* patterns, identifier* kwd_attrs, pattern* kwd_patterns)

            | MatchStar(identifier? name)
            -- The optional "rest" MatchMapping parameter handles capturing extra mapping keys

            | MatchAs(pattern? pattern, identifier? name)
            | MatchOr(pattern* patterns)

             attributes (int lineno, int col_offset, int end_lineno, int end_col_offset)
"""

def compile_match(match : ast.Match):
    print(match.subject)
    for case in reversed(match.cases):
        locals = reflect_pattern(case.pattern)
        _eval_ case.guard
        body = case.body

compile_match(mod.body[0])
print(ast.dump(mod.body[0], indent=4))

SyntaxError: positional patterns follow keyword patterns (295405050.py, line 16)

In [1]:
from kdrag.all import *

old_call = smt.FuncDeclRef.__call__


In [ ]:
#smt.FuncDeclRef.explicits = None # require them at the end for now
# Hmm. We could just have the convention that anything not fully applied is implicit
def new_call(self, *args):
    #if self.explicits is not None:
    #assert len(args) >= len(self.explicits)
    #args = args + (None,) * (self.arity() - len(args)) # support None.
    #args = [arg if arg is not None else smt.FreshConsts(self.domain()) for i,arg in enumerate(args)]
    # new_args = [args[i] if i < len(args) and args[i] is not None else smt. for i in range(self.arity())]
    if len(args) == self.arity(): # fast path
        return old_call(self, *args)
    else:
        # Maybe this should be a loop
        new_args = [args[i] \
                    if i < len(args) and args[i] is not None \
                    else smt.FreshConst(self.domain(i), prefix="?" + self.domain(i).name()) \
                    for i in range(self.arity())]
        return old_call(self, *new_args)
        #return old_call(self, *args, *[smt.FreshConst(self.domain(i), prefix="?" + self.domain(i).name()) for i in range(len(args), self.arity())])

smt.FuncDeclRef.__call__ = new_call
def implicits(t : smt.ExprRef):
    return [t for t in kd.utils.consts(t) if t.decl().name()[0] == "?"]


f = smt.Function("f", smt.IntSort(), smt.BoolSort(), smt.IntSort())
f(3)
f(3,True)
implicits(f(3))

[?Bool!140]

What about type implicits too
Wire into unify to check for "?"
kd.Meta(name, type) == "?" + name
kd.FreshMeta(type, prefix) = prefix = "?" + name

?T ?S ?W
transport(   tsubst={?T : , ...}, decls={})

?T.cast()
T = smt.DeclareSort("?T")
a = smt.TypeVar("a")
case = smt.Function("cast", a, T)

Not just consts could be `?x`?

F* tutorials



In [ ]:
def choose(T : smt.SortRef)

In [ ]:
# unbundled from the decl in tables
implicit_table = {
    decl : [("x", True), False, True, False]
}
kwarg_table = {
    decl : {"x" : 0}
}

def impl_apply(decl, *args, **kwargs):
    sig = implicit_table.get(decl)

# or tagged on each funcdecl

FuncDeclRef.implicits = None
FuncDeclRef.kwargs = None
FuncDeclRef.__call__

ExprRef.metavars = None
ExprRef.meta_constraints = None


In [ ]:
@dataclass
class App:
    f : str
    args : list["App"]
    impl_args : list["App" | None]

class App:
    def __init__(self, f, *args, **kwargs):
        self.f = f
        self.args = args
        self.kwargs = kwargs
# overwrought?

def Dummy(sort, name=None):
    if name is None:
        smt.Const("KDRAG_DUMMY", sort)
    else:
        smt.Const("DUMMY" + name, sort)

dummies = {} # ? Collect them up during construction and clear this every once in a while?







```
Ob : Type
Hom Ob Ob : Type

id {a : Ob} : Hom a a
comp (a : Ob) (b : Ob) (f : Hom a b) (g : Hom b c) : Hom a c

comp id f = f 
comp f id = f

compcomp a b c f g h : Prop
compcomp a b c f g h = f . (g . h) . k == (f . g) . h . k

% immediately reflect equals `==` into `=` if checks?

myproof a b c d e f g h : compcomp a b c f g h
myproof a b c d e f g h = cong ()

-- termination and coverage? But we have no datatypes


```

In [ ]:
# fully applied.
# no pi types, but f_ty(a,b,c) is a type that is equation to C()

# f :
#  

# def f (x : Int) (y : Int) : Int := ...
# f (x : Int) y Int : Int
# f x y = x

# f (Cons x xs) = x
from lark import Lark, Transformer, v_args
grammar = r"""

prog : stmt*

stmt : type_decl _NL | 
       eq _NL
type_decl : CNAME var_decl* ":" term
var_decl : "(" CNAME* ":" term ")"  -> expl_type
       | "{" CNAME* ":" term "}"    -> impl_type
       | term                       -> anon_type
eq : CNAME term* "=" term
term : "(" term ")" | CNAME term*

%import common.CNAME
%import common.WS_INLINE
%import common.NEWLINE -> _NL
%ignore WS_INLINE
"""

lark = Lark(grammar, start="prog", parser="lalr")
lark.parse("""
f (x : Int) : Int
g x = f (f x)
""")

UnexpectedToken: Unexpected token Token('TERMINAL', '_NL') at line 6, column 11.
Expected one of: 
	* _COLON
	* _LBRACE
	* _DOT


In [ ]:
from lark import Lark, Transformer, v_args
grammar = r"""

prog : (type_decl | eq)*
type_decl : CNAME ":" type
eq : term "=" term
term : "(" term ")" | term term | CNAME
type : "(" type ")" | "(" CNAME ":" type ")" "->" type | CNAME

%import common.CNAME
%import common.WS
%ignore WS
"""

lark = Lark(grammar, start="prog", parser="lalr")
lark.parse("""
f : (a : T) -> T
g x = f (f x)
""")
           


Tree(Token('RULE', 'prog'), [Tree(Token('RULE', 'type_decl'), [Token('CNAME', 'f'), Tree(Token('RULE', 'type'), [Token('CNAME', 'a'), Tree(Token('RULE', 'type'), [Token('CNAME', 'T')]), Tree(Token('RULE', 'type'), [Token('CNAME', 'T')])])]), Tree(Token('RULE', 'eq'), [Tree(Token('RULE', 'term'), [Tree(Token('RULE', 'term'), [Token('CNAME', 'g')]), Tree(Token('RULE', 'term'), [Token('CNAME', 'x')])]), Tree(Token('RULE', 'term'), [Tree(Token('RULE', 'term'), [Token('CNAME', 'f')]), Tree(Token('RULE', 'term'), [Tree(Token('RULE', 'term'), [Tree(Token('RULE', 'term'), [Token('CNAME', 'f')]), Tree(Token('RULE', 'term'), [Token('CNAME', 'x')])])])])])])

In [ ]:
# just get rid of objects
Arr = smt.DeclareSort("Arr")
dom = smt.Function("dom", Arr, Arr) # The identity arrow at the dom stands for
cod = smt.Function("cod", Arr, Arr)
comp = smt.Function("comp", Arr, Arr, Arr)
# id typing
dom(dom(f)) == dom(f)
cod(dom(f)) == dom(f)
dom(cod(f)) == cod(f)
cod(cod(f)) == cod(f)
comp_dom = comp(f, dom(f)) == f
cod_comp = comp(cod(f), f) == f
comp_assoc = smt.ForAll([f,g,h], cod(g) == dom(f), dom(h) == cod(g)], comp(comp(g, f), h) == comp(g, comp(f, h)))

C = smt.SetSort(Arr)

is_cat = kd.define("is_cat", [C],
    smt.ForAll([f], C[f], C[dom(f)]),
    smt.ForAll([f], C[f], C[cod(f)]),
    smt.ForAll([f,g], C[f], C[g], cod(g) == dom(f), C[comp(g, f)])
)

F = smt.Array("F", Arr, Arr)
is_functor = kd.define("is_functor", [F, C, D],
    smt.ForAll([f], C[f], D[F[f]]),
    smt.ForAll([f], C[f], F[dom(f)] == dom(F[f)]),
    smt.ForAll([f], C[f], F[cod(f)] == cod(F[f)]),
    smt.ForAll([f,g], C[f], C[g], cod(g) == dom(f), F[comp(g, f)] == comp(F[g], F[f]))
)

functor_compose = kd.prove(smt.ForAll([F,G,C,D,E], is_functor(F, C,D), is_functor(G,D,E),
    is_functor(smt.Lambda([f],G[F[f]]), C, E)
))
is_obj = kd.define("is_id" , [f], f == dom(f), f == cod(f))
hom = kd.define("hom", [C, A, B])

term = kd.define("term", [f,C],
    is_ob(f),

    smt.ForAll([g])
)














ambient wf is kind of nice.



In [ ]:
Category = kd.Struct("Category",
    ("hom", Ob, Ob, smt.SetSort(Arr)),
    ("id", smt.ArraySort(Ob, smt.BoolSort()),
    ("comp", smt.ArraySort(Arr, Arr, Arr))
))
hom = smt.Const("hom", smt.FunctionSort(Ob, Ob, smt.SetSort(Arr)))

In [ ]:
from kdrag.all import *
Arr = smt.DeclareSort("Arr")

id = smt.Function("id", Arr, Arr)
comp = smt.Function("comp", Arr, Arr, Arr)
kd.notation.matmul.register(Arr, comp)
cod = smt.Function("cod", Arr, Ob)
dom = smt.Function("dom", Arr, Ob)

C = smt.Array("C", Ob, Ob, smt.SetSort(Arr))
D = smt.Array("D", Ob, Ob, smt.SetSort(Arr))
C = smt.SetSort(Arr)
D = smt.SetSort(Arr)

is_cat = kd.define("is_cat", [C],
                   smt.And(
                    smt.ForAll([f], C[f], C[id(dom(f))], C[id(cod(f))]),   
                    smt.ForAll([f,g], C(dom(f),cod(f))[f], C(dom(g), cod(g))[g], C(dom(comp(f,g)), cod(comp(f,g)))[comp(f,g)])
                   )
                   # plays nice with id and comp
                   )


#termarrow = smt.Function("tarr", )
# terminal object
term = kd.define("term",[C,a], 
    smt.ForAll([b], smt.Exists([f], C[f], hom(b,a)[f]))
    smt.ForAll([g,f,b], C[f], hom(b,a)[f], hom(b,a)[g], g == f)
)
F = smt.ArraySort("F", Arr, Arr)
# F is a functor from C to D (it may also be a functor on different domains.
is_functor(  , [F,C,D], 
           smt.And(
               is_cat(C),
               is_cat(D),
           )
           )
# natrual trasnformations
is_nat = kd.define("is_nat", [C,D,F,G],
                   smt.And(
                          is_functor(F, C, D),
                          is_functor(G, C, D),
                   )
                   )









In [ ]:
from kdrag.all import *

Ob = smt.DeclareSort("Ob")
Arr = smt.DeclareSort("Arr")
dom = smt.Function("dom", Arr, Ob)
cod = smt.Function("cod", Arr, Ob)
a,b,c,d = smt.Consts("a b c d", Ob)
f = smt.Const("f", Arr)
#hom = smt.Function("hom", Ob, Ob, smt.SetSort(Arr))
hom = kd.define("hom", [a,b], smt.Lambda([f], smt.And(dom(f) == a, cod(f) == b)))
# hom = kd.define("hom", [a,b,f], smt.And(dom(f) == a, cod(f) == b)) ?
kd.contract(hom, [a,b,f] ,smt.And(dom(f) == a, cod(f) == b)) # Hmm. Fully applied curried contract... hmmm.

#f = smt.Const("f", hom(a,b))
f = smt.Const("f", hom(dom(f)), cod(f)) # true though by unfolding
g = smt.Const("g", hom(b,c))
h = smt.Const("h", hom(c,d))

fi = smt.Const("fi", hom(dom(f), cod(f))) # implicit type vars


id_ = smt.Function("id", Ob, Arr)

comp = smt.Function("comp", Arr, Arr, Arr)
comp_type = kd.axiom(smt.ForAll([a,b,c,f,g], hom(a,c)[f @ g]))
comp_type = kd.axiom(smt.ForAll([f,g], cod(f) == dom(g), hom(dom(f), cod(g))[comp(f,g)]))

# hmm. Not quantified enough over objects...
# since it hom(dom,cod) works. Or unfoled.
#kd.contract(comp, [f,g], smt.And(hom(a,b)[f], hom(b,c)[g]), hom(a,c)[f @ g], by=[comp_type])
kd.contract()

matmul = kd.notation.matmul.define([f,g], comp(f,g))

# Is the implicitness worth it?
kd.axiom(smt.ForAll([a,b,f], f @ id_(a) == f))
kd.axiom(smt.ForAll([a,b,f], id_(b) @ f == f))
kd.axiom(smt.ForAll([a,b,c,d,f,g,h], h @ (g @ f) == (h @ g) @ f))


S = smt.Const("S", smt.SetSort(Arr))
is_cat = kd.define("is_cat", [S], 


F = smt.Arra
is_functor = kd.define("is_functor", [F, A, B],

)





AssertionError: 

Atomic type t


Lambda : The Ultimate Pain in the Ass
Lambda free dependent type theory


Replace the complexity of lambda evaluation with the complexity of definitions.

Make an rpython interpreter

Cody said it's possible to avoid lambda, but incoherent to avoid pi.

Why do I get tripped up on Pi?


app(Vec, n)
app(Fin, n)


head : {n} -> Vec succ n -> int

head(N) : head_type(N)
has_type(head(N), head_type(N))
head_type(N) = vec(succ(N))
head(N)(cons()) = 

pi(forall X, ) ?

head_type(N) : unin(z)
univ(z) : univ(succ(z))


concat_type(N, M) = arr(vec(N), vec(M), vec(N + M))

so no pi?


Inseatd of fres hconstnatns for each new type, we could post a function `typ` tah tdoes it?


In [ ]:
Term = smt.DeclareSort("Term")
Type = smt.DeclareSort("Type")

# typ(head) is the Pi
# positing a functional relationship between terms and types is a choice.
typ_lam : smt.ArraySort(smt.ArraySort(Term, Term), smt.ArraySort(Term, Type))



In [ ]:
from typing import NamedTuple
class Defn(NamedTuple):
    name: str
    args: list[str]
    body: object






In [ ]:
import rply
lg = rply.LexerGenerator()

lg.add("LPAREN", r"\(")
lg.add("RPAREN", r"\)")
lg.add("ATOM", r"[a-zA-Z_][a-zA-Z0-9_]*")
#lg.add("NUMBER", r"[0-9]+")
lg.ignore(r"\s+")
lexer = lg.build()

pg = rply.ParserGenerator(
    # A list of all token names, accepted by the parser.
    ["LPAREN", "RPAREN", "ATOM"],
)
@pg.production("sexp : LPAREN sexp RPAREN")
def sexp_paren(p):
    return p[1:-1]

@pg.production("sexp : ATOM")
def sexp_atom(p):
    return p[0].getstr()

parser = pg.build()

def parse(s):
    tokens = lexer.lex(s)
    return parser.parse(tokens)

parse("(hello (my name) is philip)")


KeyError: 'sexp*'